# Day 14 - Capstone: End-to-End Analysis of Spotify User Behaviour  (SOLUTIONS)

**Goal:** Use everything from Days 1 to 13 on one realistic business question: who churns and why?

**Quick recap (read for 2 minutes):**

- Workflow: LOAD -> INSPECT -> CLEAN -> FEATURES -> GROUP/COMPARE -> VISUALISE -> CONCLUDE
- Outlier rule (IQR): outlier if value < Q1 - 1.5*IQR or > Q3 + 1.5*IQR
- Churn rate = share of users with status 'Inactive' (or flag = 1)
- Finish with insights in plain English: finding + number + recommended action

*There are many valid ways to solve each question. If your answer gives the same result, it is correct.*

### Warm-up (5 min)
Without looking, redo Questions 1 to 3 from **Day 13** in a scratch cell below.

### Setup (run this first)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("spotify_user_behv.csv")

### Q1. Inspect the data in one cell: shape, dtypes and missing values per column.

In [ ]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())

### Q2. What percentage of users are Inactive overall?

In [ ]:
(df["subscription_status"] == "Inactive").mean() * 100

### Q3. Compute the Inactive rate (%) per subscription type and draw it as a bar chart.

In [ ]:
(df.assign(inactive=(df["subscription_status"] == "Inactive")).groupby("subscription_type")["inactive"].mean() * 100).sort_values().plot(kind="bar")
plt.ylabel("Inactive %")
plt.title("Inactive rate by plan")
plt.show()

### Q4. Do Active and Inactive users differ in average listening hours, skips and playlists?

In [ ]:
df.groupby("subscription_status")[["avg_listening_hours_per_week", "avg_skips_per_day", "playlists_created"]].mean().round(2)

### Q5. Count the outliers in weekly listening hours with the IQR rule.

In [ ]:
q1, q3 = df["avg_listening_hours_per_week"].quantile([0.25, 0.75])
iqr = q3 - q1
mask = (df["avg_listening_hours_per_week"] < q1 - 1.5 * iqr) | (df["avg_listening_hours_per_week"] > q3 + 1.5 * iqr)
mask.sum()

### Q6. Create `hours_per_day`, `age_group` (16-24, 25-34, 35-44, 45-60), `plan_group` (Free/Paid) and `tenure_years` (as of 2026-03-31). Show 5 rows.

In [ ]:
df["hours_per_day"] = (df["avg_listening_hours_per_week"] / 7).round(2)
df["age_group"] = pd.cut(df["age"], bins=[15, 24, 34, 44, 60], labels=["16-24", "25-34", "35-44", "45-60"])
df["plan_group"] = np.where(df["subscription_type"] == "Free", "Free", "Paid")
df["signup_date"] = pd.to_datetime(df["signup_date"], format="%d-%m-%Y")
df["tenure_years"] = ((pd.Timestamp("2026-03-31") - df["signup_date"]).dt.days / 365.25).round(1)
df[["hours_per_day", "age_group", "plan_group", "tenure_years"]].head()

### Q7. Build a pivot table of Inactive rate (%) with age_group as rows and plan_group as columns.

In [ ]:
df["age_group"] = pd.cut(df["age"], bins=[15, 24, 34, 44, 60], labels=["16-24", "25-34", "35-44", "45-60"])
df["plan_group"] = np.where(df["subscription_type"] == "Free", "Free", "Paid")
df["inactive"] = (df["subscription_status"] == "Inactive").astype(int)
(pd.pivot_table(df, index="age_group", columns="plan_group", values="inactive", aggfunc="mean", observed=True) * 100).round(1)

### Q8. Ad funnel: among users who interacted with ads, what % converted to a subscription? Compare with users who did not interact.

In [ ]:
(df.assign(conv=(df["ad_conversion_to_subscription"] == "Yes")).groupby("ad_interaction")["conv"].mean() * 100).round(1)

### Q9. Which 3 countries have the highest share of paid (non-Free) users?

In [ ]:
(df.assign(paid=(df["subscription_type"] != "Free")).groupby("country")["paid"].mean() * 100).sort_values(ascending=False).head(3).round(1)

### Q10. Write 3 insights in a markdown cell: each with a finding, a number from your analysis, and one recommended business action.

In [ ]:
inactive_pct = (df["subscription_status"] == "Inactive").mean() * 100
ad_conv = (df[df["ad_interaction"] == "Yes"]["ad_conversion_to_subscription"] == "Yes").mean() * 100
top_genre = df["favorite_genre"].mode()[0]
print(f"1) {inactive_pct:.1f}% of users are Inactive -> launch a win-back campaign for them.")
print(f"2) {ad_conv:.1f}% of users who engage with ads convert -> invest in better-targeted ads.")
print(f"3) {top_genre} is the most common favourite genre -> highlight it in playlists and promotions.")